# EDA of geography and order composition

After examining delay distribution, time, and destination, two operational hypotheses are investigated: longer seller-to-customer distances may carry higher historical breach risk, and orders with more varied items, categories, or sellers may require more coordination. The target remains measured at order level.

## What can geolocation measure?

The source provides `seller_zip_code_prefix` and `customer_zip_code_prefix`, plus a table with coordinates for ZIP prefixes. The calculation has four steps: (1) aggregate repeated coordinates for each prefix by median; (2) attach coordinates to seller and customer; (3) calculate item-level origin-to-destination distance; and (4) summarize those distances at order level.

The formula is Haversine (Sinnott, 1984), using the Earth's mean radius `R = 6,371.0088 km` (Moritz, 2000):

$$
a = \sin^2\left(\frac{\Delta\varphi}{2}\right) + \cos\varphi_1 \, \cos\varphi_2 \, \sin^2\left(\frac{\Delta\lambda}{2}\right)
$$

$$
d = 2R \, \arcsin\left(\sqrt{a}\right)
$$

Where:

- $\varphi_1$, $\varphi_2$: origin (seller) and destination (customer) latitudes, in radians;
- $\Delta\varphi$: latitude difference, $\varphi_2 - \varphi_1$, in radians;
- $\Delta\lambda$: longitude difference between destination and origin, in radians;
- $R$: mean Earth radius, 6,371.0088 km;
- $a$: square of half the chord joining the two points on a sphere of unit radius;
- $d$: distance over the surface, in kilometres.

The formula treats the Earth as a sphere and computes the length of the great-circle arc between the two points, that is, the shortest path over the surface. The term $a$ combines the latitude difference with the longitude difference, the latter weighted by the cosine of the latitudes, because one degree of longitude corresponds to shorter distances away from the equator. The term $2\arcsin(\sqrt{a})$ is the central angle between the points, in radians, and multiplying by the radius converts that angle into distance. The form based on the squared sine of half the angle, the haversine, is preferred to the spherical law of cosines because it remains numerically stable for short distances (Sinnott, 1984). Because the Earth is slightly flattened at the poles, the spherical distance differs a little from the distance measured on the ellipsoid, a difference that is negligible compared with the imprecision of coordinates aggregated by ZIP prefix.

For multi-seller orders, the table also preserves minimum, mean, and maximum distance. The chart uses order mean distance. Haversine measures straight-line geographic distance, not the actual road route. If a route map is shown, it is contextual only: the historical path, hubs, transport mode, and traffic conditions of the original delivery are not available.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

LANG = "en"
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ecom_delivery_sla import (
    TABLE_FILES,
    build_order_geography,
    build_route_demo_pairs,
    build_supervised_orders,
    composition_summary,
    distance_band_summary,
    plot_composition_breach,
    plot_distance_band_breach,
    read_olist_tables,
)
from ecom_delivery_sla.maps import build_haversine_map

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'olist' / 'csv'
tables = read_olist_tables(DATA_DIR, table_names=TABLE_FILES)
supervised = build_supervised_orders(tables['orders'])
order_context = build_order_geography(
    supervised,
    tables['items'],
    tables['customers'],
    tables['sellers'],
    tables['products'],
    tables['geolocation'],
)
display(order_context[['order_id', 'distance_mean_km', 'item_count', 'distinct_product_count', 'distinct_category_count', 'seller_count']].head())

,order_id,distance_mean_km,item_count,distinct_product_count,distinct_category_count,seller_count
0,e481f51cbdc54678b7cc49136f2d6af7,18.681737,1,1,1,1
1,53cdb2fc8bc7dce0b6741e2150273451,861.036556,1,1,1,1
2,47770eb9100c2d0c44946d9cf07ec65d,514.547851,1,1,1,1
3,949d5b44dbf5de918fe9c16f97b45f8a,1821.805173,1,1,1,1
4,ad21c59c0840e6cb83a9ceb5573f8159,29.593135,1,1,1,1


### Geographic coverage insight

Approximate geolocation covers 99.51% of supervised orders. Median seller-to-customer distance is approximately 433.83 km. This provides a useful explanatory variable for EDA, but it remains an approximation based on ZIP prefixes rather than an observed road distance.

## Approximate distance and delivery breach

Orders are grouped by mean seller-to-customer distance bands. The table shows volume, breaches, breach rate, and median delay; the chart helps inspect association without implying that distance alone causes delay.

In [2]:
display(distance_band_summary(order_context))
display(plot_distance_band_breach(order_context, language=LANG))

,distance_band_km,orders,breaches,breach_rate,median_delay
0,0-100,17823,797,0.044717,-8.348715
1,100-300,13429,692,0.051530,-11.212882
2,300-600,31483,2128,0.067592,-12.369016
3,600-1000,17919,1272,0.070986,-13.149699
4,1000+,15340,1601,0.104368,-13.231586


### Distance insight

The breach rate ranges from 6.43% for approximate routes up to 100 km to 11.82% for routes above 1,000 km. Intermediate bands range from 6.37% to 8.45%. This pattern is compatible with a hypothesis of greater logistical exposure at long distances, but remains descriptive: state, category, seller, period, and order composition may explain part of the difference.

## Item, product, category, and seller variety

An order may contain multiple item rows, distinct products, categories, and sellers. These counts are calculated at order level to check whether composition is associated with promise fulfillment. This step does not assign delay to a specific seller and does not turn each item into an independent target observation.

In [3]:
display(composition_summary(order_context, dimension="items"))
display(composition_summary(order_context, dimension="products"))
display(composition_summary(order_context, dimension="category"))
display(composition_summary(order_context, dimension="sellers"))
display(plot_composition_breach(order_context, language=LANG, dimension="products"))
display(plot_composition_breach(order_context, language=LANG, dimension="sellers"))

,composition_band,orders,breaches,breach_rate,median_delay,median_items,median_sellers
0,1 item,86835,6024,0.069373,-11.507847,1.0,1.0
1,2 items,7392,397,0.053707,-12.317216,2.0,1.0
2,3 items,1306,64,0.049005,-12.404086,3.0,1.0
3,4 items,495,23,0.046465,-12.388854,4.0,1.0
4,5 items,193,9,0.046632,-13.342639,5.0,1.0
5,6+ items,249,17,0.068273,-12.268206,6.0,1.0


,composition_band,orders,breaches,breach_rate,median_delay,median_items,median_sellers
0,1 product,93273,6412,0.068744,-11.526296,1.0,1.0
1,2 products,2808,114,0.040598,-13.311470,2.0,1.0
2,3 products,298,7,0.023490,-14.109392,3.0,1.0
3,4 products,69,1,0.014493,-14.380926,4.0,1.0
4,5 products,8,0,0.000000,-12.857205,5.5,2.0
5,6+ products,14,0,0.000000,-18.102176,6.0,1.0


,composition_band,orders,breaches,breach_rate,median_delay,median_items,median_sellers
0,1 category,94415,6428,0.068082,-11.869144,1.0,1.0
1,2 categories,708,11,0.015537,-14.324769,2.0,2.0
2,3 categories,15,0,0.000000,-18.461933,3.0,3.0
3,4+ categories,0,0,NaN,NaN,NaN,NaN


,composition_band,orders,breaches,breach_rate,median_delay,median_items,median_sellers
0,1 seller,95195,6521,0.068501,-11.553600,1.0,1.0
1,2 sellers,1216,13,0.010691,-15.208119,2.0,2.0
2,3 sellers,54,0,0.000000,-18.684265,3.0,3.0
3,4+ sellers,5,0,0.000000,-20.157118,4.0,4.0


### Composition insight

Orders with one product show an 8.22% breach rate; groups with two, three, and four products show 5.06%, 4.03%, and 1.45%, respectively. Orders with two sellers represent 1,216 observations and show a 1.48% breach rate, while orders with three or more sellers are rare.

These lower rates in more complex orders do not mean that complexity reduces risk. The groups are imbalanced, may have different operational composition, and may be selected. Category variety is also rare and depends on product-category coverage. The appropriate conclusion is to keep these variables as hypotheses and evaluate them jointly with distance, period, and seller.

## Demonstration of the Haversine calculation

To make the calculation concrete, an order with one seller and one customer is selected. The map shows both points and the straight line whose distance is calculated by Haversine. Route and traffic APIs are not called because they cannot reconstruct the order's historical conditions.

In [4]:
demo_pairs = build_route_demo_pairs(
    supervised,
    tables['items'],
    tables['customers'],
    tables['sellers'],
    tables['geolocation'],
)
demo = demo_pairs.iloc[0]
display(demo[['order_id', 'seller_id', 'customer_id', 'haversine_distance_km']].to_frame('value'))
route_map = build_haversine_map(
    demo['seller_lat'],
    demo['seller_lng'],
    demo['customer_lat'],
    demo['customer_lng'],
    demo['haversine_distance_km'],
)
display(route_map)

,value
order_id,00010242fe8c5a6d1ba2dd792cb16214
seller_id,48436dade18ac8b2bce089ec2a041202
customer_id,3ce436f183e68e07877b285a838db11a
haversine_distance_km,301.229621


### Haversine demonstration insight

The dashed blue line represents the Haversine distance used in the project. The map demonstrates the origin and destination points, not the historical road path. The absence of a road route is a source limitation, not a failure of the distance calculation.

## Route network between states

This section presents the geography as a network using basic graph-theoretic concepts (Newman, 2010). Each node is a state, and each edge is a directed route, from the seller's state (origin) to the customer's state (destination). The counting unit is the order–seller pair: an order served by two sellers is counted once in each route, and the order outcome, SLA breach or not, is attributed to each of the pairs. This attribution is approximate, because the source does not say which seller caused the delay, and it is the same limitation as the order-level aggregation discussed in notebook 04. The position of each state is the median latitude and longitude of the rows of the geolocation table that belong to the state, with no external data. Routes within the same state, such as SP → SP, are not drawn as lines and appear only in the node properties.

The breach rate of a route is the share of pairs with an SLA breach. Because routes with few pairs produce unstable rates, the map draws only routes with at least 100 pairs, and the menu above the map switches this minimum volume among 100, 300 and 1,000 without re-running the cell. The uncertainty of each rate is summarised by the 95% Wilson confidence interval (Wilson, 1927), whose limits are given by:

$$
\hat{p}_{\pm} = \frac{\hat{p} + \dfrac{z^{2}}{2n} \pm z\sqrt{\dfrac{\hat{p}\,(1-\hat{p})}{n} + \dfrac{z^{2}}{4n^{2}}}}{1 + \dfrac{z^{2}}{n}}
$$

Where:

- $\hat{p}$: observed share of breaches in the route, $k/n$;
- $k$: number of pairs of the route with an SLA breach;
- $n$: number of pairs of the route;
- $z$: quantile of the standard normal distribution for the confidence level, about 1.96 for 95%.

The Wilson interval is obtained by inverting the score test for a proportion, rather than centring the interval on the observed proportion. It is therefore preferred to the usual normal approximation, $\hat{p} \pm z\sqrt{\hat{p}(1-\hat{p})/n}$, when $n$ is small or the rate is close to zero, a common situation for low-volume routes with a low breach rate: the normal approximation can produce limits outside the range 0 to 1 and a zero-width interval when no breach is observed, whereas the Wilson interval stays within the range 0 to 1 and keeps coverage close to the nominal level.

The map is meant to be read with the cursor. Each edge has an arrow at its midpoint that indicates the direction of the route, and the line width grows with volume while the colour represents the breach rate. Hovering over the arrow shows the origin, the destination, the number of pairs, the breaches, the rate with its 95% interval, the share of the route in the total volume, the median Haversine distance and the medians of the promised and actual lead times. Each node has a size proportional to the total volume of pairs involving it and the colour of the breach rate of the orders received; its hover shows the pairs sent and received, the outbound and inbound rates, the pairs within the state itself, the numbers of sellers and customers, the out-degree and in-degree (number of routes above the minimum volume) and the strength (summed volume of those routes).

The rate of a route mixes destination and period, because the delay peaks of November 2017 and of February and March 2018 are not equally distributed across routes. The reading is therefore descriptive and not causal. The graph is the visual counterpart of the as-of seller → state network features used by the model, with the difference that here the whole period is used, whereas the model features use only the history preceding each order.

In [5]:
from ecom_delivery_sla_viz import (
    plot_route_matrix,
    plot_route_network,
    plot_top_city_routes,
    route_pairs,
    route_table,
)

route_pair_table = route_pairs(
    supervised,
    tables['items'],
    tables['sellers'],
    tables['customers'],
    tables['geolocation'],
)
display(route_table(route_pair_table, level="state", min_orders=100).head(10))
display(plot_route_network(route_pair_table, tables['geolocation'], language=LANG))

,origin,destination,origin_state,destination_state,orders,breaches,breach_rate,ci_low,ci_high,share,distance_median_km,promised_median_days,actual_median_days,is_self_loop
0,SP,SP,SP,SP,31180,1431,0.045895,0.043628,0.048274,0.318778,94.967364,17.150017,6.518657,True
1,SP,RJ,SP,RJ,8274,1157,0.139836,0.132530,0.147476,0.084592,383.754544,24.421858,12.766146,False
2,SP,MG,SP,MG,7562,394,0.052103,0.047317,0.057343,0.077312,488.720494,23.434583,10.770683,False
3,SP,RS,SP,RS,3662,238,0.064992,0.057455,0.073441,0.037440,867.538356,28.673304,13.827477,False
4,SP,PR,SP,PR,3164,138,0.043616,0.037035,0.051303,0.032348,419.257758,24.293831,11.066348,False
5,PR,SP,PR,SP,2975,90,0.030252,0.024677,0.037038,0.030416,469.147835,24.617072,9.431586,False
6,MG,SP,MG,SP,2581,91,0.035258,0.028805,0.043092,0.026388,467.009680,22.458275,9.195069,False
7,SP,SC,SP,SC,2355,222,0.094268,0.083118,0.106739,0.024077,544.157927,25.279525,13.831308,False
8,SP,BA,SP,BA,2340,297,0.126923,0.114044,0.141025,0.023924,1448.330267,29.301134,17.584491,False
9,MG,MG,MG,MG,1542,39,0.025292,0.018556,0.034387,0.015765,239.548351,19.402141,7.263767,True


### Insight on the route graph

The network is a star centred on SP. Of the ten largest routes between states, seven leave SP, and the largest is the internal route SP → SP, with 31,180 pairs, 31.9% of the total, and a breach rate of 4.6% (95% CI from 4.4% to 4.8%). The largest routes arriving in SP come from PR and MG, with rates of 3.0% and 3.5%.

For the same origin, the rate varies widely with the destination. From SP, it is 4.4% to PR, 5.2% to MG and 6.5% to RS, but 9.4% to SC, 12.7% to BA and 14.0% to RJ. Distance alone does not explain this difference: SP → RJ has a median distance of 384 km, shorter than SP → MG (489 km), and still has almost three times the breach rate. The promised lead time does not offset it either, because the median promise to RJ (24.4 days) is close to the one to MG (23.4 days). SP → BA combines the largest distance in the table (1,448 km), the longest median actual lead time (17.6 days) and a rate of 12.7%. The internal routes SP → SP and MG → MG have the lowest rates, 4.6% and 2.5%. The reading is descriptive: each route's rate mixes destination and period and does not identify the cause of the delay.

## Origin × destination matrix

The matrix complements the map by showing, for the eight origin states with the highest volume (rows) and for the 27 destination states (columns), the breach rate of each route by colour and the number of pairs written in the cell. Unlike the map, the matrix includes the diagonal, that is, the pairs within the state itself, and routes below 100 pairs. To limit instability, cells with fewer than 30 pairs are left blank. The unit remains the order–seller pair, and the hover shows the same properties as the map edge, including the 95% Wilson interval (Wilson, 1927). The rate of each cell mixes destination and period, and the reading is descriptive.

In [6]:
display(route_table(route_pair_table, level="state", min_orders=30).head(10))
display(plot_route_matrix(route_pair_table, language=LANG))

,origin,destination,origin_state,destination_state,orders,breaches,breach_rate,ci_low,ci_high,share,distance_median_km,promised_median_days,actual_median_days,is_self_loop
0,SP,SP,SP,SP,31180,1431,0.045895,0.043628,0.048274,0.318778,94.967364,17.150017,6.518657,True
1,SP,RJ,SP,RJ,8274,1157,0.139836,0.132530,0.147476,0.084592,383.754544,24.421858,12.766146,False
2,SP,MG,SP,MG,7562,394,0.052103,0.047317,0.057343,0.077312,488.720494,23.434583,10.770683,False
3,SP,RS,SP,RS,3662,238,0.064992,0.057455,0.073441,0.037440,867.538356,28.673304,13.827477,False
4,SP,PR,SP,PR,3164,138,0.043616,0.037035,0.051303,0.032348,419.257758,24.293831,11.066348,False
5,PR,SP,PR,SP,2975,90,0.030252,0.024677,0.037038,0.030416,469.147835,24.617072,9.431586,False
6,MG,SP,MG,SP,2581,91,0.035258,0.028805,0.043092,0.026388,467.009680,22.458275,9.195069,False
7,SP,SC,SP,SC,2355,222,0.094268,0.083118,0.106739,0.024077,544.157927,25.279525,13.831308,False
8,SP,BA,SP,BA,2340,297,0.126923,0.114044,0.141025,0.023924,1448.330267,29.301134,17.584491,False
9,MG,MG,MG,MG,1542,39,0.025292,0.018556,0.034387,0.015765,239.548351,19.402141,7.263767,True


### Insight on the origin × destination matrix

The table that accompanies the matrix repeats the largest routes, and the matrix adds the smaller ones. In the SP row, the darkest cells are RJ and North-East destinations with enough volume, such as MA (487 pairs) and AL (257 pairs), while the South and the South-East itself stay light. The AL column is also dark from PR (36 pairs), and the CE column from RJ (54 pairs), which suggests a destination effect repeated across origins. These volumes are small, and the Wilson interval shown on hover is wide. Many cells of the smaller origins are blank because they have fewer than 30 pairs, which indicates lack of volume, not absence of delay.

## Largest routes between cities

Between cities, the number of routes is much larger and most have few pairs, so a complete graph would be unreadable and its rates would be unstable. For this reason, cities appear only in this chart, which shows the 15 city → city routes with the highest volume. Each bar represents the breach rate of the route, and the error bar represents the 95% Wilson interval (Wilson, 1927): wide intervals indicate less precise rates. The hover shows the number of pairs, the breaches, the interval and the median Haversine distance. City names follow the source registry, without accents, and spelling variants of the same city are treated as distinct cities, which can fragment the volume of some routes.

In [7]:
display(route_table(route_pair_table, level="city", min_orders=1).head(15))
display(plot_top_city_routes(route_pair_table, language=LANG))

,origin,destination,origin_state,origin_city,destination_state,destination_city,orders,breaches,breach_rate,ci_low,ci_high,share,distance_median_km,promised_median_days,actual_median_days,is_self_loop
0,Sao Paulo (SP),Sao Paulo (SP),SP,sao paulo,SP,sao paulo,4090,169,0.041320,0.035639,0.047863,0.041815,13.603608,13.309606,4.592691,True
1,Sao Paulo (SP),Rio De Janeiro (RJ),SP,sao paulo,RJ,rio de janeiro,1380,160,0.115942,0.100106,0.133910,0.014109,346.260872,24.330851,11.613183,False
2,Ibitinga (SP),Sao Paulo (SP),SP,ibitinga,SP,sao paulo,1125,64,0.056889,0.044801,0.071993,0.011502,302.260873,20.267072,9.147083,False
3,Sao Paulo (SP),Belo Horizonte (MG),SP,sao paulo,MG,belo horizonte,575,39,0.067826,0.050010,0.091378,0.005879,490.205259,21.308449,9.005787,False
4,Ibitinga (SP),Rio De Janeiro (RJ),SP,ibitinga,RJ,rio de janeiro,490,80,0.163265,0.133181,0.198588,0.005010,581.362689,24.032147,14.440995,False
5,Sao Paulo (SP),Brasilia (DF),SP,sao paulo,DF,brasilia,472,21,0.044492,0.029282,0.067056,0.004826,873.957844,22.375411,10.138461,False
6,Santo Andre (SP),Sao Paulo (SP),SP,santo andre,SP,sao paulo,426,19,0.044601,0.028736,0.068605,0.004355,18.395674,14.452170,4.373860,False
7,Curitiba (PR),Sao Paulo (SP),PR,curitiba,SP,sao paulo,369,9,0.024390,0.012884,0.045697,0.003773,340.375969,22.312650,7.199653,False
8,Sao Paulo (SP),Campinas (SP),SP,sao paulo,SP,campinas,364,32,0.087912,0.062960,0.121471,0.003721,89.622159,13.619606,6.073148,False
9,Sao Paulo (SP),Curitiba (PR),SP,sao paulo,PR,curitiba,352,9,0.025568,0.013509,0.047871,0.003599,342.408683,21.473264,8.866047,False


### Insight on the city routes

The largest city route is internal to São Paulo, with 4,090 pairs, a median distance of 14 km and a rate of 4.1%. São Paulo → Rio de Janeiro has 1,380 pairs and a rate of 11.6% (95% CI from 10.0% to 13.4%), and Ibitinga → Rio de Janeiro, with 490 pairs, reaches 16.3% (CI from 13.3% to 19.9%), which reproduces at city level the RJ pattern seen between states. São Paulo → Belo Horizonte has 575 pairs and a rate of 6.8% (CI from 5.0% to 9.1%), and São Paulo → Salvador has 319 pairs and 14.4%.

Routes with 300 to 500 pairs have intervals several percentage points wide, such as Curitiba → São Paulo, from 1.3% to 4.6%, so the ranking between routes with close rates should not be read as a real difference. The presence of Ibitinga and other inland São Paulo cities among the largest origins shows that shipments do not originate only in state capitals.

## Limits and next step

This analysis does not measure the actual route, historical traffic, partial delivery, or an individual seller SLA. It produces order-level context measures: approximate distance and composition.

The next step is to turn approved decisions into an analytical table with one row per `order_id`, including coverage, aggregation, and temporal-availability controls. Until the contracts are complete, `production_eligible = false`.

## References

Moritz, H. (2000). Geodetic Reference System 1980. *Journal of Geodesy*, 74(1), 128–133. https://doi.org/10.1007/s001900050278

Sinnott, R. W. (1984). Virtues of the Haversine. *Sky and Telescope*, 68(2), 159.

Newman, M. E. J. (2010). *Networks: An Introduction*. Oxford University Press.

Wilson, E. B. (1927). Probable inference, the law of succession, and statistical inference. *Journal of the American Statistical Association*, 22(158), 209–212. https://doi.org/10.1080/01621459.1927.10502953